# 📰 Fake News Detection System
## Complete Machine Learning Pipeline

**Author:** AI/ML Engineer  
**Stack:** Python · pandas · scikit-learn · NLTK  

---

### What This Notebook Does
1. Load and merge `Fake.csv` + `True.csv`
2. Clean and preprocess text (NLP pipeline)
3. Vectorise with TF-IDF
4. Train 3 ML models and compare accuracies
5. Show confusion matrix and classification report
6. Save the best model + vectoriser to disk
7. Demo: Predict new articles

---

## 0. Install / Import Dependencies

In [ ]:
# Install packages (run once)
# !pip install pandas scikit-learn nltk matplotlib seaborn

import sys
import re
import string
import pickle
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# Download NLTK resources
for resource in ['stopwords', 'punkt', 'punkt_tab']:
    nltk.download(resource, quiet=True)

print('✓ All imports successful')

## 1. Load and Merge Datasets

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Update these paths if your CSV files are in a different location
FAKE_PATH = Path('../data/Fake.csv')
TRUE_PATH = Path('../data/True.csv')

# ── Load ───────────────────────────────────────────────────────────────────
df_fake = pd.read_csv(FAKE_PATH)
df_true = pd.read_csv(TRUE_PATH)

print(f'Fake articles : {len(df_fake):,}')
print(f'Real articles : {len(df_true):,}')

df_fake.head(3)

In [ ]:
# ── Label & Merge ─────────────────────────────────────────────────────────
# Fake = 0, Real = 1  (binary classification convention)
df_fake['label'] = 0
df_true['label'] = 1

df = (
    pd.concat([df_fake, df_true], ignore_index=True)
    .sample(frac=1, random_state=42)   # shuffle
    .reset_index(drop=True)
)

# Combine title + text into a single 'content' column for richer signal
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

print(f'Total articles: {len(df):,}')
print('\nLabel distribution:')
print(df['label'].value_counts())
df.head(3)

## 2. NLP Preprocessing

We clean the text through 5 steps:

| Step | What it does | Why |
|------|--------------|-----|
| Lowercase | `News → news` | `News` and `news` are the same word |
| Remove URLs | `http://... → ` | Links add noise, not signal |
| Remove punctuation | `fact! → fact` | Punctuation doesn't help classify |
| Remove stopwords | Remove `the`, `is`, `at` | No meaning in fake vs real |
| Stemming | `running → run` | Different forms = same concept |

In [ ]:
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Full NLP preprocessing pipeline for a single string."""
    if not isinstance(text, str):
        return ''
    
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 4. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'[^a-z\s]', '', text)
    # 5. Tokenise, remove stopwords, stem
    tokens = [
        stemmer.stem(word)
        for word in text.split()
        if word not in stop_words and len(word) > 2
    ]
    return ' '.join(tokens)

# Demo
sample = 'BREAKING! Scientists are RUNNING experiments and discovering AMAZING facts!!!'
print('Original :', sample)
print('Cleaned  :', clean_text(sample))

In [ ]:
# Apply to entire dataset
print('Preprocessing... (may take 1–2 minutes)')
t0 = time.time()
df['cleaned_text'] = df['content'].apply(clean_text)
print(f'Done in {time.time()-t0:.1f}s')

# Preview
df[['content', 'cleaned_text', 'label']].head(3)

## 3. TF-IDF Vectorisation

**TF-IDF explained simply:**

Computers can't understand words — only numbers.  
TF-IDF converts each article into a vector of numbers.

- **TF** (Term Frequency): How often does `conspiracy` appear in *this* article?
- **IDF** (Inverse Doc Frequency): Is `conspiracy` rare across *all* articles? If so, it's more informative.

Result: Words that are common in one article AND rare across the whole dataset get high scores — exactly what separates fake from real news vocabulary.

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────
X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training : {len(X_train):,} samples')
print(f'Testing  : {len(X_test):,} samples')

# ── TF-IDF ────────────────────────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=50_000,   # Keep top 50k most informative words
    ngram_range=(1, 2),    # Include word pairs (bigrams) like 'fake news'
    sublinear_tf=True,     # Apply log scaling to term frequencies
    min_df=2,              # Ignore words appearing in < 2 documents
)

# IMPORTANT: fit_transform on TRAIN only — never use test data to fit!
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'\nVocabulary size : {len(vectorizer.vocabulary_):,}')
print(f'Matrix shape    : {X_train_tfidf.shape}')

## 4. Train Multiple Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1, random_state=42
    ),
    'Passive Aggressive': PassiveAggressiveClassifier(
        max_iter=50, C=1.0, random_state=42, n_jobs=-1
    ),
    'Naive Bayes': MultinomialNB(alpha=0.1),
}

results  = {}
trained  = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    elapsed = time.time() - t0
    
    results[name]  = {'accuracy': acc, 'y_pred': y_pred}
    trained[name]  = model
    
    print(f'{name:35} Accuracy: {acc:.4f}  ({elapsed:.2f}s)')

## 5. Evaluate — Confusion Matrix & Classification Report

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = ['Fake', 'Real']

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, annot_kws={'size': 13, 'weight': 'bold'}
    )
    ax.set_title(f'{name}\nAccuracy: {res["accuracy"]:.4f}', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full classification report for each model
for name, res in results.items():
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(classification_report(y_test, res['y_pred'], target_names=class_names))

## 6. Save Best Model and Vectoriser

In [ ]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

# Find best model
best_name  = max(results, key=lambda n: results[n]['accuracy'])
best_model = trained[best_name]
best_acc   = results[best_name]['accuracy']

print(f'Best model : {best_name}  ({best_acc:.4f})')

# Save model
with open(MODELS_DIR / 'best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f, protocol=pickle.HIGHEST_PROTOCOL)

# Save vectoriser (MUST be saved separately — it holds the vocabulary!)
with open(MODELS_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f, protocol=pickle.HIGHEST_PROTOCOL)

print('✓ Model saved    → models/best_model.pkl')
print('✓ Vectoriser saved → models/tfidf_vectorizer.pkl')

## 7. Predict New Articles

In [ ]:
# Load saved artefacts (simulates production usage)
with open(MODELS_DIR / 'best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open(MODELS_DIR / 'tfidf_vectorizer.pkl', 'rb') as f:
    loaded_vectorizer = pickle.load(f)

def predict_article(text: str) -> dict:
    cleaned  = clean_text(text)
    features = loaded_vectorizer.transform([cleaned])
    label    = int(loaded_model.predict(features)[0])
    if hasattr(loaded_model, 'predict_proba'):
        confidence = float(loaded_model.predict_proba(features)[0][label])
    else:
        score = float(loaded_model.decision_function(features)[0])
        confidence = float(1 / (1 + np.exp(-abs(score))))
    return {
        'verdict':    'REAL' if label == 1 else 'FAKE',
        'confidence': round(confidence, 4),
    }

# Test predictions
test_articles = [
    'SHOCKING: Deep state operatives control weather to destroy crops!!!',
    'Congress approves $500 billion infrastructure package after bipartisan vote.',
    'Scientists discover aliens have been living underground for thousands of years.',
    'Federal Reserve raises interest rates by 0.25 percent amid inflation concerns.',
]

for article in test_articles:
    result = predict_article(article)
    icon = '🔴' if result['verdict'] == 'FAKE' else '🟢'
    print(f"{icon} {result['verdict']} ({result['confidence']:.1%}) — {article[:70]}")

---
## 🚀 Improvements & Future Upgrades

### Better Accuracy (same tech stack)
| Idea | Expected Impact |
|------|-----------------|
| Hyperparameter tuning (`GridSearchCV`) | +1–3% |
| SVM with linear kernel | +1–2% |
| Ensemble (voting classifier) | +1–3% |
| Feature engineering (source metadata) | +2–5% |
| Character n-grams in TF-IDF | +0.5–1% |

### Deep Learning / BERT Upgrade
```python
# Replace TF-IDF + sklearn with:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='ProsusAI/finbert'   # or 'jy46604790/Fake-News-Bert-Detect'
)
result = classifier('Breaking: Scientists discover moon is hollow...')
```

**Why BERT is better:**
- Understands **context** ("bank" in finance vs. river)
- Pre-trained on 3.3 billion words
- State-of-the-art on most NLP benchmarks
- Fine-tuning takes ~30 min on GPU vs. hours of feature engineering